# Forecast `deals_next_quarter`: simplified simulation-oriented architecture

В этом ноутбуке строится новая архитектура прогноза квартальных продаж `deals_next_q`,
ориентированная на последующее использование в Monte Carlo.

Ключевые изменения:
- класс жилья **не отбрасывается** при малом числе наблюдений;
- неизвестный класс проекта заполняется через **сопоставление средней цены проекта со средними рыночными ценами по классам и по годам**;
- модель для условного среднего \(\mu_{i}\) упрощена до **состояния проекта + параметров проекта**, без лагов продаж;
- сравниваются 4 модели:
  1. `NB_offset`
  2. `Poisson_offset`
  3. `NB_feature_remaining_lots`
  4. `Poisson_feature_remaining_lots`
- делаются два блока экспериментов:
  1. одна общая модель на всех классах сразу;
  2. отдельная модель внутри каждого класса.


In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from scipy.optimize import minimize
from scipy.special import betaln, expit
from scipy.stats import betabinom, nbinom, poisson

warnings.filterwarnings("ignore")
pd.options.display.max_columns = 250
pd.options.display.width = 260
plt.style.use("seaborn-v0_8-whitegrid")


def show_table(df: pd.DataFrame, caption: str | None = None, round_map: dict | None = None):
    if caption:
        print(f"\n{caption}")
    out = df.copy()
    if round_map:
        for col, digits in round_map.items():
            if col in out.columns:
                out[col] = out[col].round(digits)
    display(out)


In [2]:
REGION_KEY = "Краснодар"

DATA_DIR = Path("..") / "data"
DEALS_DIR = DATA_DIR / "deals_parquet"
PROJECTS_DIR = DATA_DIR / "projects_parquet"


def find_region_parquet_files(folder: Path, region_key: str | None) -> list[Path]:
    files = sorted(folder.glob("*.parquet"))
    if region_key:
        key = region_key.lower()
        files = [p for p in files if key in p.name.lower()]
    return files


DEALS_PARQUET_FILES = find_region_parquet_files(DEALS_DIR, REGION_KEY)
PROJECTS_PARQUET_FILES = find_region_parquet_files(PROJECTS_DIR, REGION_KEY)

if not DEALS_PARQUET_FILES:
    raise FileNotFoundError(f"No deals parquet files for region: {REGION_KEY}")
if not PROJECTS_PARQUET_FILES:
    raise FileNotFoundError(f"No projects parquet files for region: {REGION_KEY}")

print("Region:", REGION_KEY)
print("Deals files:", len(DEALS_PARQUET_FILES))
for p in DEALS_PARQUET_FILES:
    print(" -", p)
print("Projects files:", len(PROJECTS_PARQUET_FILES))
for p in PROJECTS_PARQUET_FILES:
    print(" -", p)


Region: Краснодар
Deals files: 1
 - ..\data\deals_parquet\bnMAP_pro_Сделки_Краснодар_11-02-2026_12-20_part1.parquet
Projects files: 1
 - ..\data\projects_parquet\bnMAP_pro_ПД_Краснодар_11-02-2026_15-13_part1.parquet


## 1) Построение квартальной витрины `project-quarter`

Ключевая логика exposure:
- `total_lots`: число уникальных лотов в проекте (из ПД);
- `cum_sales_prev_q`: накопленные продажи до начала квартала;
- `remaining_lots_start_q = total_lots - cum_sales_prev_q` (с ограничением снизу `1`);
- в модели NB этот exposure идет в `offset` как `log(exposure)`.


In [3]:
def to_datetime_series(series: pd.Series) -> pd.Series:
    x = series.replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    parsed = pd.to_datetime(x, errors="coerce", dayfirst=True, utc=True)
    if hasattr(parsed, "dt"):
        parsed = parsed.dt.tz_localize(None)
    num = pd.to_numeric(x, errors="coerce")
    excel_dt = pd.to_datetime(num, unit="D", origin="1899-12-30", errors="coerce")
    return parsed.fillna(excel_dt)


def to_float_series(series: pd.Series) -> pd.Series:
    x = (
        series.astype(str)
        .str.replace("\xa0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
        .replace({"nan": np.nan, "None": np.nan, "": np.nan})
    )
    return pd.to_numeric(x, errors="coerce")


def normalize_class(x: object) -> str:
    if x is None or pd.isna(x):
        return "Неизвестно"
    s = str(x).strip().lower()
    if "комфорт" in s:
        return "Комфорт"
    if "премиум" in s:
        return "Премиум"
    if "бизнес" in s:
        return "Бизнес"
    if "эконом" in s:
        return "Эконом"
    if not s:
        return "Неизвестно"
    return str(x).strip().title()


def normalize_room_type(x: object) -> str:
    if x is None or pd.isna(x):
        return "other"
    s = str(x).strip().lower()
    if s in {"ст", "studio", "0", "студия"}:
        return "studio"
    if s in {"1", "1к", "1-комн", "1 комн"}:
        return "1r"
    if s in {"2", "2к", "3", "3к", "4", "4к", "5", "5к"}:
        return "2plus"
    return "other"


def mode_or_unknown(s: pd.Series) -> str:
    vals = s.dropna().astype(str).str.strip()
    if vals.empty:
        return "Неизвестно"
    m = vals.mode()
    return m.iloc[0] if not m.empty else vals.iloc[0]


In [4]:
import pyarrow.parquet as pq

DEALS_COLS = [
    "ID проекта",
    "Дата договора",
    "Цена за кв. метр",
    "Площадь согласно ЕГРН",
    "Универсальная комнатность",
    "Класс",
]
PROJECTS_COLS = [
    "ID проекта",
    "ID лота",
    "Класс",
    "Район",
    "Конструктив",
    "Старт продаж",
    "Плановая дата РВЭ",
]


def read_parquet_with_normalized_columns(path: Path, required_cols: list[str]) -> pd.DataFrame:
    schema_cols = pq.ParquetFile(path).schema.names
    by_norm = {str(c).strip(): c for c in schema_cols}
    actual_cols = [by_norm[c] for c in required_cols if c in by_norm]

    part = pd.read_parquet(path, columns=actual_cols if actual_cols else None)
    part = part.rename(columns=lambda c: str(c).strip())

    for c in required_cols:
        if c not in part.columns:
            part[c] = pd.NA
    return part[required_cols]


def read_many_parquet(paths: list[Path], required_cols: list[str], label: str) -> pd.DataFrame:
    frames = []
    for p in paths:
        part = read_parquet_with_normalized_columns(p, required_cols)
        print(f"{label}: {p.name} -> {part.shape}")
        frames.append(part)
    out = pd.concat(frames, axis=0, ignore_index=True)
    print(f"{label} total shape: {out.shape}")
    return out


deals = read_many_parquet(DEALS_PARQUET_FILES, DEALS_COLS, "deals").copy()
projects = read_many_parquet(PROJECTS_PARQUET_FILES, PROJECTS_COLS, "projects").copy()

projects = projects.drop_duplicates(subset=["ID проекта", "ID лота"]).reset_index(drop=True)

deals["project_id"] = deals["ID проекта"].astype("string").str.strip()
projects["project_id"] = projects["ID проекта"].astype("string").str.strip()

deals = deals[deals["project_id"].notna() & (deals["project_id"] != "")]
projects = projects[projects["project_id"].notna() & (projects["project_id"] != "")]

deals["deal_date"] = to_datetime_series(deals["Дата договора"])
deals = deals[deals["deal_date"].notna()].copy()
deals["quarter"] = deals["deal_date"].dt.to_period("Q")

deals["price_sqm"] = to_float_series(deals["Цена за кв. метр"])
deals["area_sqm"] = to_float_series(deals["Площадь согласно ЕГРН"])
deals["room_type"] = deals["Универсальная комнатность"].map(normalize_room_type)
deals["class_group"] = deals["Класс"].map(normalize_class)
projects["start_sales_dt"] = to_datetime_series(projects["Старт продаж"])
projects["planned_rve_dt"] = to_datetime_series(projects["Плановая дата РВЭ"])

deals.head(3)


deals: bnMAP_pro_Сделки_Краснодар_11-02-2026_12-20_part1.parquet -> (234896, 6)
deals total shape: (234896, 6)
projects: bnMAP_pro_ПД_Краснодар_11-02-2026_15-13_part1.parquet -> (505639, 7)
projects total shape: (505639, 7)


,ID проекта,Дата договора,Цена за кв. метр,Площадь согласно ЕГРН,Универсальная комнатность,Класс,project_id,deal_date,quarter,price_sqm,area_sqm,room_type,class_group
0,dded6592573a9a248fcc6a73d36ae678,2013-06-28,<NA>,63.40,2,Комфорт,dded6592573a9a248fcc6a73d36ae678,2013-06-28,2013Q2,NaN,63.4,2plus,Комфорт
1,dded6592573a9a248fcc6a73d36ae678,2013-07-01,57000,37.00,1,Комфорт,dded6592573a9a248fcc6a73d36ae678,2013-07-01,2013Q3,57000.0,37.0,1r,Комфорт
2,dded6592573a9a248fcc6a73d36ae678,2013-07-01,<NA>,32.00,1,Комфорт,dded6592573a9a248fcc6a73d36ae678,2013-07-01,2013Q3,NaN,32.0,1r,Комфорт


## 1) Построение квартальной витрины и заполнение класса жилья

Логика заполнения `class_group`:

1. Берём известные классы из сделок и/или ПД.
2. Для каждого года считаем рыночную среднюю цену за кв.м. по классам.
3. Для проекта с неизвестным классом считаем среднюю цену за кв.м. по проекту в каждом году.
4. Для каждого года выбираем класс с ближайшей годовой средней ценой.
5. По проекту берём моду годовых присвоений. Если годовых наблюдений нет, используем fallback по общей средней цене проекта.


In [5]:

# ---------- 1.1. Подготовка сделок и рыночных годовых ориентиров по классам ----------

deals["deal_year"] = deals["deal_date"].dt.year.astype("Int64")

# Сначала попробуем получить проектный класс из сделок, затем из ПД.
deal_class_by_project = (
    deals.groupby("project_id", as_index=False)
    .agg(class_from_deals=("class_group", mode_or_unknown))
)
deal_class_by_project["class_from_deals"] = deal_class_by_project["class_from_deals"].map(normalize_class)

projects["class_group_norm"] = projects["Класс"].map(normalize_class)
proj_total_lots = (
    projects.groupby("project_id", as_index=False)["ID лота"]
    .nunique()
    .rename(columns={"ID лота": "total_lots"})
)
proj_meta = (
    projects.groupby("project_id", as_index=False)
    .agg(
        class_from_projects=("class_group_norm", mode_or_unknown),
        district=("Район", mode_or_unknown),
        building_type=("Конструктив", mode_or_unknown),
        start_sales=("start_sales_dt", "min"),
        planned_rve=("planned_rve_dt", "min"),
    )
)

proj_meta["class_from_projects"] = proj_meta["class_from_projects"].map(normalize_class)
proj_meta["district"] = proj_meta["district"].fillna("Неизвестно").astype(str)
proj_meta["building_type"] = proj_meta["building_type"].fillna("Неизвестно").astype(str)
proj_meta["start_sales"] = to_datetime_series(proj_meta["start_sales"])
proj_meta["planned_rve"] = to_datetime_series(proj_meta["planned_rve"])

proj_meta = proj_meta.merge(deal_class_by_project, on="project_id", how="left")
proj_meta["class_group"] = proj_meta["class_from_projects"]
mask_unknown = proj_meta["class_group"].isna() | (proj_meta["class_group"] == "Неизвестно")
proj_meta.loc[mask_unknown, "class_group"] = proj_meta.loc[mask_unknown, "class_from_deals"]

known_market = deals.loc[
    deals["class_group"].notna()
    & (deals["class_group"] != "Неизвестно")
    & deals["price_sqm"].notna()
    & deals["deal_year"].notna(),
    ["deal_year", "class_group", "price_sqm"]
].copy()

year_class_market = (
    known_market.groupby(["deal_year", "class_group"], as_index=False)
    .agg(
        market_mean_price_sqm=("price_sqm", "mean"),
        market_median_price_sqm=("price_sqm", "median"),
        market_obs=("price_sqm", "size"),
    )
)

overall_class_market = (
    known_market.groupby("class_group", as_index=False)
    .agg(
        market_mean_price_sqm=("price_sqm", "mean"),
        market_median_price_sqm=("price_sqm", "median"),
        market_obs=("price_sqm", "size"),
    )
)

project_year_price = (
    deals.loc[deals["price_sqm"].notna() & deals["deal_year"].notna(), ["project_id", "deal_year", "price_sqm"]]
    .groupby(["project_id", "deal_year"], as_index=False)
    .agg(project_year_mean_price_sqm=("price_sqm", "mean"), project_year_obs=("price_sqm", "size"))
)

project_overall_price = (
    deals.loc[deals["price_sqm"].notna(), ["project_id", "price_sqm"]]
    .groupby("project_id", as_index=False)
    .agg(project_mean_price_sqm=("price_sqm", "mean"), project_obs=("price_sqm", "size"))
)


def infer_class_by_yearly_price(project_id: str) -> str:
    sub = project_year_price.loc[project_year_price["project_id"] == project_id].copy()
    if not sub.empty:
        candidates = sub.merge(year_class_market, on="deal_year", how="left")
        candidates = candidates.dropna(subset=["market_mean_price_sqm"])
        if not candidates.empty:
            candidates["abs_diff"] = (candidates["project_year_mean_price_sqm"] - candidates["market_mean_price_sqm"]).abs()
            best_per_year = candidates.sort_values(["deal_year", "abs_diff", "market_obs"]).groupby("deal_year", as_index=False).first()
            votes = best_per_year["class_group"].astype(str)
            if not votes.empty:
                return votes.mode().iloc[0]

    sub_all = project_overall_price.loc[project_overall_price["project_id"] == project_id].copy()
    if not sub_all.empty and not overall_class_market.empty:
        target = float(sub_all["project_mean_price_sqm"].iloc[0])
        cand = overall_class_market.copy()
        cand["abs_diff"] = (cand["market_mean_price_sqm"] - target).abs()
        cand = cand.sort_values(["abs_diff", "market_obs"], ascending=[True, False])
        return str(cand.iloc[0]["class_group"])

    return "Неизвестно"


need_infer = proj_meta["class_group"].isna() | (proj_meta["class_group"] == "Неизвестно")
proj_meta.loc[need_infer, "class_group"] = proj_meta.loc[need_infer, "project_id"].map(infer_class_by_yearly_price)
proj_meta["class_group"] = proj_meta["class_group"].fillna("Неизвестно").astype(str)

class_fill_summary = pd.DataFrame(
    {
        "projects_total": [len(proj_meta)],
        "projects_with_known_class_initial": [int((~need_infer).sum())],
        "projects_filled_by_price_matching_or_fallback": [int(need_infer.sum())],
        "projects_still_unknown_after_fill": [int((proj_meta["class_group"] == "Неизвестно").sum())],
    }
)
show_table(class_fill_summary, "Сводка по заполнению класса проекта")

show_table(
    year_class_market.sort_values(["deal_year", "class_group"]).head(20),
    "Пример годовых рыночных средних цен по классам",
    {"market_mean_price_sqm": 0, "market_median_price_sqm": 0},
)



Сводка по заполнению класса проекта


,projects_total,projects_with_known_class_initial,projects_filled_by_price_matching_or_fallback,projects_still_unknown_after_fill
0,195,194,1,1



Пример годовых рыночных средних цен по классам


,deal_year,class_group,market_mean_price_sqm,market_median_price_sqm,market_obs
0,2013,Бизнес,163000.0,163000.0,1
1,2013,Комфорт,98692.0,118000.0,13
2,2014,Бизнес,164000.0,164000.0,1
3,2014,Комфорт,164823.0,185000.0,31
4,2015,Комфорт,139493.0,137500.0,80
5,2015,Эконом,52000.0,52000.0,2
6,2016,Бизнес,156333.0,156333.0,1
7,2016,Комфорт,127267.0,137500.0,80
8,2016,Эконом,44000.0,44000.0,15
9,2017,Комфорт,148053.0,138750.0,16


In [6]:

# ---------- 1.2. Построение квартальной panel-таблицы ----------

# Quarterly aggregates from deals (features at quarter q)
q_sales = (
    deals.groupby(["project_id", "quarter"], as_index=False)
    .agg(
        deals_q=("project_id", "size"),
        avg_price_sqm_q=("price_sqm", "mean"),
        median_price_sqm_q=("price_sqm", "median"),
        avg_area_q=("area_sqm", "mean"),
    )
)
q_sales["year"] = q_sales["quarter"].dt.year.astype(int)

# Build complete panel between first and last observed quarter for each project
bounds = q_sales.groupby("project_id", as_index=False).agg(q_min=("quarter", "min"), q_max=("quarter", "max"))
rows = []
for r in bounds.itertuples(index=False):
    for q in pd.period_range(r.q_min, r.q_max, freq="Q"):
        rows.append((r.project_id, q))
panel = pd.DataFrame(rows, columns=["project_id", "quarter"])
panel["year"] = panel["quarter"].dt.year.astype(int)

panel = panel.merge(q_sales, on=["project_id", "quarter", "year"], how="left")
panel["deals_q"] = panel["deals_q"].fillna(0).astype(int)

panel = panel.merge(proj_total_lots, on="project_id", how="left")
panel = panel.merge(
    proj_meta[["project_id", "class_group", "district", "building_type", "start_sales", "planned_rve"]],
    on="project_id",
    how="left",
)

# Fill static fields
panel["class_group"] = panel["class_group"].fillna("Неизвестно")
panel["district"] = panel["district"].fillna("Неизвестно")
panel["building_type"] = panel["building_type"].fillna("Неизвестно")

# Numerical fillings
for c in ["avg_price_sqm_q", "median_price_sqm_q", "avg_area_q"]:
    panel[c] = panel.groupby("project_id")[c].ffill()
    panel[c] = panel.groupby("project_id")[c].bfill()

# market yearly references
year_market_mean = (
    deals.loc[deals["price_sqm"].notna() & deals["deal_year"].notna()]
    .groupby("deal_year", as_index=False)
    .agg(market_price_sqm_year=("price_sqm", "mean"))
    .rename(columns={"deal_year": "year"})
)
year_class_market_features = (
    year_class_market[["deal_year", "class_group", "market_mean_price_sqm"]]
    .rename(columns={"deal_year": "year"})
    .copy()
)
panel = panel.merge(year_market_mean, on="year", how="left")
panel = panel.merge(
    year_class_market_features.rename(columns={"market_mean_price_sqm": "market_price_sqm_year_class"}),
    on=["year", "class_group"],
    how="left",
)

# fallback fills
global_price = float(deals["price_sqm"].dropna().median())
global_area = float(deals["area_sqm"].dropna().median())
global_market_year = float(year_market_mean["market_price_sqm_year"].dropna().median()) if not year_market_mean.empty else global_price

for c, fallback in [
    ("avg_price_sqm_q", global_price),
    ("median_price_sqm_q", global_price),
    ("avg_area_q", global_area),
    ("market_price_sqm_year", global_market_year),
]:
    panel[c] = pd.to_numeric(panel[c], errors="coerce").fillna(fallback)

panel["market_price_sqm_year_class"] = pd.to_numeric(panel["market_price_sqm_year_class"], errors="coerce")
panel["market_price_sqm_year_class"] = panel["market_price_sqm_year_class"].fillna(panel["market_price_sqm_year"])

# Exposure / inventory state
panel = panel.sort_values(["project_id", "quarter"]).reset_index(drop=True)
panel["total_lots"] = pd.to_numeric(panel["total_lots"], errors="coerce").fillna(0).astype(int)
lots_fallback = int(panel.loc[panel["total_lots"] > 0, "total_lots"].median()) if (panel["total_lots"] > 0).any() else 1
panel["total_lots"] = panel["total_lots"].replace(0, lots_fallback)

panel["cum_sales_prev_q"] = panel.groupby("project_id")["deals_q"].cumsum().shift(1).fillna(0)
panel["remaining_lots_start_q"] = (panel["total_lots"] - panel["cum_sales_prev_q"]).clip(lower=1)
panel["remaining_inventory_share"] = (panel["remaining_lots_start_q"] / panel["total_lots"]).clip(0, 1)
panel["share_sold_already"] = 1 - panel["remaining_inventory_share"]

# Project age / time to delivery
panel["start_q"] = to_datetime_series(panel["start_sales"]).dt.to_period("Q")
panel["rve_q"] = to_datetime_series(panel["planned_rve"]).dt.to_period("Q")
panel["q_ord"] = panel["quarter"].map(lambda p: p.ordinal)
panel["start_q_ord"] = panel["start_q"].map(lambda p: p.ordinal if pd.notna(p) else np.nan)
panel["rve_q_ord"] = panel["rve_q"].map(lambda p: p.ordinal if pd.notna(p) else np.nan)
panel["project_age_q"] = (panel["q_ord"] - panel["start_q_ord"] + 1).clip(lower=0)
panel["quarters_to_delivery"] = panel["rve_q_ord"] - panel["q_ord"]

panel["project_age_q"] = panel["project_age_q"].fillna(panel["project_age_q"].median())
panel["quarters_to_delivery"] = panel["quarters_to_delivery"].fillna(panel["quarters_to_delivery"].median())
panel["quarter_num"] = panel["quarter"].dt.quarter.astype(int)

# Price-position features relative to market / class
panel["price_to_market_year_ratio"] = panel["avg_price_sqm_q"] / panel["market_price_sqm_year"].replace(0, np.nan)
panel["price_to_class_year_ratio"] = panel["avg_price_sqm_q"] / panel["market_price_sqm_year_class"].replace(0, np.nan)
panel["price_gap_to_market_year"] = panel["avg_price_sqm_q"] - panel["market_price_sqm_year"]
panel["price_gap_to_class_year"] = panel["avg_price_sqm_q"] - panel["market_price_sqm_year_class"]

for c in ["price_to_market_year_ratio", "price_to_class_year_ratio"]:
    panel[c] = panel[c].replace([np.inf, -np.inf], np.nan).fillna(1.0)

for c in ["price_gap_to_market_year", "price_gap_to_class_year"]:
    panel[c] = panel[c].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Target: next quarter deals
panel["deals_next_q"] = panel.groupby("project_id")["deals_q"].shift(-1)
model_df = panel[panel["deals_next_q"].notna()].copy()
model_df["deals_next_q"] = model_df["deals_next_q"].astype(int)

summary = pd.DataFrame(
    {
        "rows_total": [len(model_df)],
        "projects": [model_df["project_id"].nunique()],
        "min_quarter": [str(model_df["quarter"].min())],
        "max_quarter": [str(model_df["quarter"].max())],
        "mean_target_deals_next_q": [model_df["deals_next_q"].mean()],
        "mean_remaining_lots_start_q": [model_df["remaining_lots_start_q"].mean()],
    }
)
show_table(summary, "Параметры итоговой обучающей витрины", {"mean_target_deals_next_q": 2, "mean_remaining_lots_start_q": 2})

class_summary = (
    model_df.groupby("class_group", as_index=False)
    .agg(
        rows=("project_id", "size"),
        projects=("project_id", "nunique"),
        mean_deals_next_q=("deals_next_q", "mean"),
        mean_price_sqm=("avg_price_sqm_q", "mean"),
    )
    .sort_values("rows", ascending=False)
)
show_table(class_summary, "Состав витрины по классам", {"mean_deals_next_q": 2, "mean_price_sqm": 0})



Параметры итоговой обучающей витрины


,rows_total,projects,min_quarter,max_quarter,mean_target_deals_next_q,mean_remaining_lots_start_q
0,2829,167,2009Q3,2025Q4,81.71,2875.52



Состав витрины по классам


,class_group,rows,projects,mean_deals_next_q,mean_price_sqm
1,Комфорт,1789,104,110.31,114334.0
2,Эконом,722,42,30.70,83558.0
0,Бизнес,310,20,37.07,156473.0
3,Элитный,8,1,19.38,290762.0


## 2) Общие функции моделирования и метрики

В обеих ветках используются только признаки, которые можно интерпретировать как состояние проекта
или параметры проекта, пригодные для последующей Monte Carlo-симуляции.


In [7]:

def project_level_me_per_lot(project_id, total_lots, y_true: np.ndarray, y_pred: np.ndarray) -> float:
    df = pd.DataFrame(
        {
            "project_id": pd.Series(project_id, copy=False),
            "total_lots": pd.to_numeric(pd.Series(total_lots, copy=False), errors="coerce"),
            "error": np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float),
        }
    )
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["project_id", "total_lots", "error"])
    if df.empty:
        return np.nan
    df["total_lots"] = df["total_lots"].clip(lower=1.0)
    proj = (
        df.groupby("project_id", as_index=False)
        .agg(me=("error", "mean"), total_lots=("total_lots", "max"))
    )
    return float((proj["me"] / proj["total_lots"]).mean())


def _poisson_nll(y: np.ndarray, mu: np.ndarray) -> float:
    y = np.asarray(y, dtype=float)
    mu = np.asarray(mu, dtype=float).clip(1e-12, None)
    return float(np.sum(mu - y * np.log(mu) + np.where(y > 0, np.log(np.arange(1, 2)).sum(), 0.0)))


def poisson_loglik_sum(y: np.ndarray, mu: np.ndarray) -> float:
    from scipy.special import gammaln
    y = np.asarray(y, dtype=float)
    mu = np.asarray(mu, dtype=float).clip(1e-12, None)
    return float(np.sum(y * np.log(mu) - mu - gammaln(y + 1.0)))


def nb_loglik_sum(y: np.ndarray, mu: np.ndarray, alpha: float) -> float:
    from scipy.special import gammaln
    y = np.asarray(y, dtype=float)
    mu = np.asarray(mu, dtype=float).clip(1e-12, None)
    a = float(max(alpha, 1e-12))
    if a <= 1e-8:
        return poisson_loglik_sum(y, mu)
    r = 1.0 / a
    p = r / (r + mu)
    ll = gammaln(y + r) - gammaln(r) - gammaln(y + 1.0) + r * np.log(p) + y * np.log1p(-p)
    return float(np.sum(np.where(np.isfinite(ll), ll, -1e100)))


def betabin_loglik_sum(y: np.ndarray, n: np.ndarray, pi: np.ndarray, kappa: float) -> float:
    from scipy.special import gammaln
    y = np.asarray(y, dtype=float)
    n = np.asarray(n, dtype=float)
    pi = np.asarray(pi, dtype=float).clip(1e-8, 1 - 1e-8)
    kappa = float(np.clip(kappa, 1e-6, 1e6))
    a = pi * kappa
    b = (1.0 - pi) * kappa
    ll = (
        gammaln(n + 1.0)
        - gammaln(y + 1.0)
        - gammaln(n - y + 1.0)
        + betaln(y + a, n - y + b)
        - betaln(a, b)
    )
    return float(np.sum(np.where(np.isfinite(ll), ll, -1e100)))


def distribution_chi2(y_true: np.ndarray, mu: np.ndarray, dist: str, alpha: float | None = None, n_trials: np.ndarray | None = None, kappa: float | None = None):
    y = np.asarray(y_true, dtype=int)
    mu = np.asarray(mu, dtype=float).clip(1e-9, None)
    if len(y) == 0:
        return np.nan, 0, 0

    if n_trials is not None:
        n_trials = np.asarray(n_trials, dtype=int)
        if len(n_trials) != len(y):
            n_trials = None

    k_max = int(max(20, min(int(y.max()), int(np.quantile(y, 0.99)) + 5)))
    obs = np.bincount(np.minimum(y, k_max), minlength=k_max + 1).astype(float)

    exp_main = []
    if dist == "poisson":
        for k in range(k_max):
            exp_main.append(float(np.sum(poisson.pmf(k, mu))))
    elif dist == "nb":
        a = float(max(alpha if alpha is not None else 0.3, 1e-9))
        if a <= 1e-4:
            for k in range(k_max):
                exp_main.append(float(np.sum(poisson.pmf(k, mu))))
        else:
            r = 1.0 / a
            p = r / (r + mu)
            for k in range(k_max):
                vals = nbinom.pmf(k, r, p)
                vals = np.where(np.isfinite(vals), vals, 0.0)
                exp_main.append(float(np.sum(vals)))
    elif dist == "betabinomial":
        if n_trials is None:
            return np.nan, 0, k_max
        kk = float(np.clip(kappa if kappa is not None else 30.0, 1e-6, 1e6))
        pi = np.clip(mu / np.clip(n_trials, 1, None), 1e-8, 1 - 1e-8)
        a = pi * kk
        b = (1.0 - pi) * kk
        for k in range(k_max):
            valid = n_trials >= k
            vals = np.zeros(len(y), dtype=float)
            if np.any(valid):
                vals[valid] = betabinom.pmf(k, n_trials[valid], a[valid], b[valid])
            vals = np.where(np.isfinite(vals), vals, 0.0)
            exp_main.append(float(np.sum(vals)))
    else:
        return np.nan, 0, k_max

    exp_main = np.array(exp_main, dtype=float)
    tail = float(len(y) - np.sum(exp_main))
    exp = np.concatenate([exp_main, [max(tail, 1e-9)]])
    mask = exp >= 1.0
    chi2 = float(np.sum(((obs[mask] - exp[mask]) ** 2) / exp[mask])) if mask.any() else np.nan
    return chi2, int(mask.sum()), k_max


# Более узкий и устойчивый набор фич.
# Убираем шумные / дублирующие признаки:
# - district, building_type, year: слишком много разреженных dummy
# - share_sold_already: линейно связан с remaining_inventory_share
# - price_gap_* и второй ratio: дублируют относительную цену
# - avg_area_q: часто шумит и мало добавляет к стабильности fit
SIM_NUMERIC_BASE = [
    "project_age_q",
    "quarters_to_delivery",
    "remaining_inventory_share",
]
SIM_CAT_BASE = ["class_group", "quarter_num"]


def prepare_design_sim(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    remaining_lots_mode: str = "offset",  # "offset" or "feature"
    include_class_feature: bool = True,
):
    tr = train_df.copy()
    te = test_df.copy()

    # Устойчивые лог-трансформации масштабирующих признаков.
    for df in (tr, te):
        df["log_avg_price_sqm_q"] = np.log(np.clip(pd.to_numeric(df["avg_price_sqm_q"], errors="coerce"), 1.0, None))
        df["log_total_lots"] = np.log(np.clip(pd.to_numeric(df["total_lots"], errors="coerce"), 1.0, None))
        df["log_price_to_market_year_ratio"] = np.log(
            np.clip(pd.to_numeric(df["price_to_market_year_ratio"], errors="coerce"), 1e-3, 1e3)
        )
        df["quarters_to_delivery"] = pd.to_numeric(df["quarters_to_delivery"], errors="coerce").clip(-8, 20)

    numeric_cols = SIM_NUMERIC_BASE + [
        "log_avg_price_sqm_q",
        "log_total_lots",
        "log_price_to_market_year_ratio",
    ]
    cat_cols = SIM_CAT_BASE.copy()

    if not include_class_feature and "class_group" in cat_cols:
        cat_cols.remove("class_group")

    if remaining_lots_mode == "feature":
        tr["log_remaining_lots_feat"] = np.log(np.clip(pd.to_numeric(tr["remaining_lots_start_q"], errors="coerce"), 1.0, None))
        te["log_remaining_lots_feat"] = np.log(np.clip(pd.to_numeric(te["remaining_lots_start_q"], errors="coerce"), 1.0, None))
        numeric_cols = numeric_cols + ["log_remaining_lots_feat"]

    for c in numeric_cols:
        tr[c] = pd.to_numeric(tr[c], errors="coerce")
        te[c] = pd.to_numeric(te[c], errors="coerce")

    tr[numeric_cols] = tr[numeric_cols].replace([np.inf, -np.inf], np.nan)
    te[numeric_cols] = te[numeric_cols].replace([np.inf, -np.inf], np.nan)

    med = tr[numeric_cols].median()
    tr[numeric_cols] = tr[numeric_cols].fillna(med)
    te[numeric_cols] = te[numeric_cols].fillna(med)

    # winsor + standardize
    for c in numeric_cols:
        lo = float(tr[c].quantile(0.01))
        hi = float(tr[c].quantile(0.99))
        if not np.isfinite(lo):
            lo = -1e9
        if not np.isfinite(hi):
            hi = 1e9
        if lo > hi:
            lo, hi = hi, lo
        tr[c] = tr[c].clip(lo, hi)
        te[c] = te[c].clip(lo, hi)

        mean = float(tr[c].mean())
        std = float(tr[c].std())
        if not np.isfinite(std) or std < 1e-8:
            std = 1.0
        tr[c] = (tr[c] - mean) / std
        te[c] = (te[c] - mean) / std
        tr[c] = tr[c].clip(-6, 6)
        te[c] = te[c].clip(-6, 6)

    for c in cat_cols:
        tr[c] = tr[c].astype("string").fillna("UNK")
        te[c] = te[c].astype("string").fillna("UNK")
        top_vals = tr[c].value_counts().head(12).index
        tr[c] = tr[c].where(tr[c].isin(top_vals), "OTHER")
        te[c] = te[c].where(te[c].isin(top_vals), "OTHER")

    x_train = pd.get_dummies(tr[numeric_cols + cat_cols], columns=cat_cols, drop_first=True, dtype=float)
    x_test = pd.get_dummies(te[numeric_cols + cat_cols], columns=cat_cols, drop_first=True, dtype=float)
    x_test = x_test.reindex(columns=x_train.columns, fill_value=0.0)

    col_var = x_train.var(axis=0)
    keep_cols = col_var[col_var > 1e-10].index.tolist()
    if not keep_cols:
        keep_cols = x_train.columns.tolist()
    x_train = x_train[keep_cols]
    x_test = x_test[keep_cols]

    # Убираем избыточно коррелированные признаки.
    corr = x_train.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_cols = [c for c in upper.columns if any(upper[c] > 0.995)]
    if drop_cols:
        x_train = x_train.drop(columns=drop_cols)
        x_test = x_test.drop(columns=drop_cols, errors="ignore")

    x_train = sm.add_constant(x_train, has_constant="add").astype(float)
    x_test = sm.add_constant(x_test, has_constant="add").astype(float)

    y_train = pd.to_numeric(tr["deals_next_q"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    y_test = pd.to_numeric(te["deals_next_q"], errors="coerce").fillna(0.0).to_numpy(dtype=float)

    exp_train = pd.to_numeric(tr["remaining_lots_start_q"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(1.0).clip(lower=1.0)
    exp_test = pd.to_numeric(te["remaining_lots_start_q"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(1.0).clip(lower=1.0)
    offset_train = np.log(exp_train.to_numpy(dtype=float))
    offset_test = np.log(exp_test.to_numpy(dtype=float))

    return x_train, x_test, y_train, y_test, offset_train, offset_test


def fit_poisson_glm_safe(y, X, offset=None, label="Poisson"):
    y_arr = np.asarray(y, dtype=float)
    max_mu_ok = float(max(1e4, (np.nanmax(y_arr) if y_arr.size else 1.0) * 100.0))
    fit_specs = [
        ("irls", "fit", {"maxiter": 300}),
        ("ridge_1e-5", "fit_regularized", {"alpha": 1e-5, "L1_wt": 0.0, "maxiter": 3000}),
        ("ridge_1e-4", "fit_regularized", {"alpha": 1e-4, "L1_wt": 0.0, "maxiter": 3000}),
        ("ridge_1e-3", "fit_regularized", {"alpha": 1e-3, "L1_wt": 0.0, "maxiter": 3000}),
        ("ridge_1e-2", "fit_regularized", {"alpha": 1e-2, "L1_wt": 0.0, "maxiter": 3000}),
    ]

    best = None
    last_exc = None
    for fit_name, method_name, kwargs in fit_specs:
        try:
            model = sm.GLM(y_arr, X, family=sm.families.Poisson(), offset=offset)
            res = getattr(model, method_name)(**kwargs)
            fitted_mu = np.asarray(res.predict(X, offset=offset), dtype=float)
            if np.all(np.isfinite(fitted_mu)) and float(np.nanmax(fitted_mu)) <= max_mu_ok:
                ll = poisson_loglik_sum(y_arr, fitted_mu)
                cand = (ll, res, fit_name)
                if (best is None) or (cand[0] > best[0]):
                    best = cand
            else:
                last_exc = RuntimeError(f"unstable fitted values for {label} using {fit_name}")
        except Exception as exc:
            last_exc = exc
    if best is not None:
        return best[1], best[2]
    raise RuntimeError(f"{label} failed: {last_exc}")


def fit_nb_glm_safe(y, X, offset=None, alpha_init=0.1, label="NB"):
    y_arr = np.asarray(y, dtype=float)
    max_mu_ok = float(max(1e4, (np.nanmax(y_arr) if y_arr.size else 1.0) * 100.0))

    alpha_candidates = []
    base_candidates = [alpha_init, alpha_init * 0.5, alpha_init * 1.5, 0.03, 0.05, 0.1, 0.2, 0.3, 0.7, 1.5]
    for a in base_candidates:
        a = float(max(min(a, 5.0), 1e-5))
        if a not in alpha_candidates:
            alpha_candidates.append(a)

    reg_candidates = [None, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2]
    last_exc = None
    best = None

    poisson_start = None
    try:
        pois_res, _ = fit_poisson_glm_safe(y_arr, X, offset=offset, label=f"{label}_warmstart")
        poisson_start = np.asarray(pois_res.params, dtype=float)
    except Exception:
        poisson_start = None

    for a in alpha_candidates:
        family = sm.families.NegativeBinomial(alpha=a)
        for reg in reg_candidates:
            try:
                model = sm.GLM(y_arr, X, family=family, offset=offset)
                if reg is None:
                    fit_kwargs = {"maxiter": 500}
                    if poisson_start is not None and len(poisson_start) == X.shape[1]:
                        fit_kwargs["start_params"] = poisson_start
                    res = model.fit(**fit_kwargs)
                    fit_name = f"irls_alpha={a:.4g}"
                else:
                    fit_kwargs = {"alpha": reg, "L1_wt": 0.0, "maxiter": 4000}
                    if poisson_start is not None and len(poisson_start) == X.shape[1]:
                        fit_kwargs["start_params"] = poisson_start
                    res = model.fit_regularized(**fit_kwargs)
                    fit_name = f"ridge_alphaNB={a:.4g}_pen={reg:.4g}"

                fitted_mu = np.asarray(res.predict(X, offset=offset), dtype=float)
                if np.all(np.isfinite(fitted_mu)) and float(np.nanmax(fitted_mu)) <= max_mu_ok:
                    ll = nb_loglik_sum(y_arr, fitted_mu, a)
                    cand = (ll, res, a, fit_name)
                    if (best is None) or (cand[0] > best[0]):
                        best = cand
                else:
                    last_exc = RuntimeError(f"{label}: unstable fitted values for {fit_name}")
            except Exception as exc:
                last_exc = exc

    if best is not None:
        return best[1], best[2], best[3]
    raise RuntimeError(f"{label} failed for all NB fits: {last_exc}")


def fit_betabinomial_glm_safe(y, n_trials, X, label="BetaBinomial"):
    y_arr = np.asarray(y, dtype=float)
    n_arr = np.asarray(n_trials, dtype=float)
    n_arr = np.maximum(n_arr, y_arr)
    n_arr = np.clip(n_arr, 1.0, None)

    # Стартуем с binomial-GLM по доле продаж.
    start_beta = np.zeros(X.shape[1], dtype=float)
    try:
        frac = np.clip(y_arr / n_arr, 1e-6, 1 - 1e-6)
        binom_mod = sm.GLM(frac, X, family=sm.families.Binomial(), var_weights=n_arr)
        binom_res = binom_mod.fit(maxiter=300)
        start_beta = np.asarray(binom_res.params, dtype=float)
    except Exception:
        pass

    kappa_candidates = [5.0, 10.0, 20.0, 40.0, 80.0, 150.0]
    reg_candidates = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2]
    start_shift_grid = [0.0, -0.5, 0.5]

    best = None
    last_exc = None

    def build_obj(Xm, yv, nv, reg):
        def obj(theta):
            beta = theta[:-1]
            log_kappa = float(theta[-1])
            eta = Xm @ beta
            pi = expit(np.clip(eta, -30, 30))
            kappa = float(np.exp(np.clip(log_kappa, -10, 12)))
            ll = betabin_loglik_sum(yv, nv, pi, kappa)
            pen = reg * float(np.sum(beta[1:] ** 2))
            return -ll + pen
        return obj

    for kappa0 in kappa_candidates:
        for reg in reg_candidates:
            obj = build_obj(np.asarray(X, dtype=float), y_arr, n_arr, reg)
            for shift in start_shift_grid:
                theta0 = np.concatenate([start_beta.copy(), [np.log(kappa0)]])
                theta0[0] += shift
                try:
                    res = minimize(
                        obj,
                        theta0,
                        method="L-BFGS-B",
                        options={"maxiter": 2000, "ftol": 1e-10, "maxls": 50},
                    )
                    if (not res.success) and (not np.isfinite(res.fun)):
                        continue
                    theta_hat = np.asarray(res.x, dtype=float)
                    beta_hat = theta_hat[:-1]
                    kappa_hat = float(np.exp(np.clip(theta_hat[-1], -10, 12)))
                    pi_hat = expit(np.clip(np.asarray(X, dtype=float) @ beta_hat, -30, 30))
                    mu_hat = n_arr * pi_hat
                    if np.all(np.isfinite(mu_hat)) and np.nanmax(mu_hat) <= np.nanmax(n_arr) + 1e-9:
                        ll = betabin_loglik_sum(y_arr, n_arr, pi_hat, kappa_hat)
                        cand = (ll, beta_hat, kappa_hat, reg, mu_hat)
                        if (best is None) or (cand[0] > best[0]):
                            best = cand
                except Exception as exc:
                    last_exc = exc

    if best is None:
        raise RuntimeError(f"{label} failed: {last_exc}")

    class BetaBinomResult:
        def __init__(self, beta, kappa):
            self.params = np.asarray(beta, dtype=float)
            self.kappa = float(kappa)

        def predict_mu(self, X_new, n_trials_new):
            X_new = np.asarray(X_new, dtype=float)
            n_trials_new = np.asarray(n_trials_new, dtype=float)
            pi = expit(np.clip(X_new @ self.params, -30, 30))
            return np.asarray(n_trials_new, dtype=float) * pi

        def predict_pi(self, X_new):
            X_new = np.asarray(X_new, dtype=float)
            return expit(np.clip(X_new @ self.params, -30, 30))

    return BetaBinomResult(best[1], best[2]), best[2], f"lbfgs_kappa={best[2]:.4g}_pen={best[3]:.4g}"


def chronological_split_60_40(df: pd.DataFrame):
    # Разбиение по времени, но по накопленной доле строк, а не по числу кварталов.
    q_counts = (
        df.groupby("quarter", as_index=False)
        .size()
        .rename(columns={"size": "rows"})
        .sort_values("quarter")
        .reset_index(drop=True)
    )
    if len(q_counts) <= 1:
        uniq_q = sorted(df["quarter"].unique())
        return df.copy(), df.iloc[0:0].copy(), uniq_q, []

    q_counts["cum_share"] = q_counts["rows"].cumsum() / q_counts["rows"].sum()
    cut_pos = int(np.searchsorted(q_counts["cum_share"].to_numpy(dtype=float), 0.60, side="left"))
    cut_pos = max(0, min(cut_pos, len(q_counts) - 2))

    train_q = q_counts.loc[:cut_pos, "quarter"].tolist()
    test_q = q_counts.loc[cut_pos + 1 :, "quarter"].tolist()

    train_df = df[df["quarter"].isin(train_q)].copy()
    test_df = df[df["quarter"].isin(test_q)].copy()
    return train_df, test_df, train_q, test_q


def collect_metrics(name, y_true, y_pred, project_id, total_lots, dist_kind=None, alpha=None, n_trials=None, kappa=None):
    out = {
        "model": name,
        "ME_per_lot": project_level_me_per_lot(project_id, total_lots, y_true, y_pred),
    }
    chi2, bins_used = np.nan, 0
    if dist_kind is not None:
        chi2, bins_used, _ = distribution_chi2(y_true, y_pred, dist=dist_kind, alpha=alpha, n_trials=n_trials, kappa=kappa)
    out["chi2_distribution_per_bin"] = float(chi2 / bins_used) if bins_used > 0 and np.isfinite(chi2) else np.nan
    out["chi2_bins_used"] = bins_used
    return out


def _extract_model_param_series(fitted_result, feature_names):
    feature_names = [str(c) for c in feature_names]
    out = pd.Series(np.nan, index=feature_names, dtype=float)
    params = getattr(fitted_result, "params", None)
    if params is None:
        return out
    if isinstance(params, pd.Series):
        params = pd.to_numeric(params, errors="coerce")
        params.index = params.index.astype(str)
        return params.reindex(feature_names).astype(float)
    arr = np.asarray(params, dtype=float).reshape(-1)
    n_copy = min(len(arr), len(feature_names))
    out.iloc[:n_copy] = arr[:n_copy]
    return out


def collect_model_weights_wide(fitted_result, model_name, train_class, feature_names, alpha=None, kappa=None):
    coef = _extract_model_param_series(fitted_result, feature_names)
    row = {
        "model_name": str(model_name),
        "train_class": str(train_class),
        "__alpha__": float(alpha) if alpha is not None and np.isfinite(alpha) else np.nan,
        "__kappa__": float(kappa) if kappa is not None and np.isfinite(kappa) else np.nan,
    }
    for feature_name in feature_names:
        value = coef.loc[str(feature_name)]
        row[str(feature_name)] = float(value) if pd.notna(value) else np.nan
    return row


def register_model_weight_columns(feature_names, feature_name_order, feature_name_seen):
    for feature_name in feature_names:
        feature_name = str(feature_name)
        if feature_name not in feature_name_seen:
            feature_name_seen.add(feature_name)
            feature_name_order.append(feature_name)


def build_weights_wide_df(weight_rows, feature_name_order):
    ordered_feature_cols = [c for c in feature_name_order if c != "const"]
    if "const" in feature_name_order:
        ordered_feature_cols = ["const"] + ordered_feature_cols
    final_cols = ["model_name", "train_class"] + ordered_feature_cols + ["__alpha__", "__kappa__"]
    if not weight_rows:
        return pd.DataFrame(columns=final_cols)
    weights_df = pd.DataFrame(weight_rows)
    return weights_df.reindex(columns=final_cols)


def run_four_models(train_df, test_df, include_class_feature=True, train_class="all_classes", model_weight_rows=None, model_weight_feature_order=None, model_weight_feature_seen=None):
    x_train_off, x_test_off, y_train, y_test, off_train, off_test = prepare_design_sim(
        train_df, test_df, remaining_lots_mode="offset", include_class_feature=include_class_feature
    )
    x_train_feat, x_test_feat, _, _, _, _ = prepare_design_sim(
        train_df, test_df, remaining_lots_mode="feature", include_class_feature=include_class_feature
    )
    off_feature_names = x_train_off.columns.astype(str).tolist()
    feat_feature_names = x_train_feat.columns.astype(str).tolist()
    collect_weights = (
        model_weight_rows is not None
        and model_weight_feature_order is not None
        and model_weight_feature_seen is not None
    )

    y_mean = float(np.mean(y_train))
    y_var = float(np.var(y_train, ddof=1)) if len(y_train) > 1 else y_mean
    alpha_hat = max((y_var - y_mean) / max(y_mean**2, 1e-9), 1e-4)
    alpha_hat = float(min(alpha_hat, 5.0))

    n_train_bb = np.maximum(
        pd.to_numeric(train_df["remaining_lots_start_q"], errors="coerce").fillna(1.0).to_numpy(dtype=float),
        y_train,
    )
    n_test_bb = np.maximum(
        pd.to_numeric(test_df["remaining_lots_start_q"], errors="coerce").fillna(1.0).to_numpy(dtype=float),
        y_test,
    )

    preds = pd.DataFrame(
        {
            "project_id": test_df["project_id"].values,
            "quarter": test_df["quarter"].astype(str).values,
            "class_group": test_df["class_group"].values,
            "y_true": y_test,
            "total_lots": pd.to_numeric(test_df["total_lots"], errors="coerce").fillna(1.0).values.astype(float),
            "remaining_lots_start_q": test_df["remaining_lots_start_q"].values.astype(float),
        }
    )

    metrics = []

    nb_off_res, alpha_nb_off, nb_off_fit_name = fit_nb_glm_safe(
        y_train, x_train_off, offset=off_train, alpha_init=alpha_hat, label="NB_offset"
    )
    print(f"[INFO] NB_offset used {nb_off_fit_name}")
    mu_nb_off = np.asarray(np.clip(nb_off_res.predict(x_test_off, offset=off_test), 1e-9, None), dtype=float)
    preds["pred_nb_offset"] = mu_nb_off
    metrics.append(
        collect_metrics(
            "NB_offset", y_test, mu_nb_off, preds["project_id"].values, preds["total_lots"].values, dist_kind="nb", alpha=alpha_nb_off
        )
    )
    if collect_weights:
        register_model_weight_columns(off_feature_names, model_weight_feature_order, model_weight_feature_seen)
        model_weight_rows.append(
            collect_model_weights_wide(
                nb_off_res,
                "NB_offset",
                train_class,
                off_feature_names,
                alpha=alpha_nb_off,
            )
        )

    pois_off, pois_off_fit_name = fit_poisson_glm_safe(
        y_train, x_train_off, offset=off_train, label="Poisson_offset"
    )
    if pois_off_fit_name != "irls":
        print(f"[INFO] Poisson_offset used {pois_off_fit_name}")
    mu_pois_off = np.asarray(np.clip(pois_off.predict(x_test_off, offset=off_test), 1e-9, None), dtype=float)
    preds["pred_poisson_offset"] = mu_pois_off
    metrics.append(
        collect_metrics(
            "Poisson_offset", y_test, mu_pois_off, preds["project_id"].values, preds["total_lots"].values, dist_kind="poisson"
        )
    )
    if collect_weights:
        register_model_weight_columns(off_feature_names, model_weight_feature_order, model_weight_feature_seen)
        model_weight_rows.append(
            collect_model_weights_wide(
                pois_off,
                "Poisson_offset",
                train_class,
                off_feature_names,
            )
        )

    nb_feat_res, alpha_nb_feat, nb_feat_fit_name = fit_nb_glm_safe(
        y_train, x_train_feat, offset=None, alpha_init=alpha_hat, label="NB_feature_remaining_lots"
    )
    print(f"[INFO] NB_feature_remaining_lots used {nb_feat_fit_name}")
    mu_nb_feat = np.asarray(np.clip(nb_feat_res.predict(x_test_feat), 1e-9, None), dtype=float)
    preds["pred_nb_feature_remaining_lots"] = mu_nb_feat
    metrics.append(
        collect_metrics(
            "NB_feature_remaining_lots", y_test, mu_nb_feat, preds["project_id"].values, preds["total_lots"].values, dist_kind="nb", alpha=alpha_nb_feat
        )
    )
    if collect_weights:
        register_model_weight_columns(feat_feature_names, model_weight_feature_order, model_weight_feature_seen)
        model_weight_rows.append(
            collect_model_weights_wide(
                nb_feat_res,
                "NB_feature_remaining_lots",
                train_class,
                feat_feature_names,
                alpha=alpha_nb_feat,
            )
        )

    pois_feat, pois_feat_fit_name = fit_poisson_glm_safe(
        y_train, x_train_feat, offset=None, label="Poisson_feature_remaining_lots"
    )
    if pois_feat_fit_name != "irls":
        print(f"[INFO] Poisson_feature_remaining_lots used {pois_feat_fit_name}")
    mu_pois_feat = np.asarray(np.clip(pois_feat.predict(x_test_feat), 1e-9, None), dtype=float)
    preds["pred_poisson_feature_remaining_lots"] = mu_pois_feat
    metrics.append(
        collect_metrics(
            "Poisson_feature_remaining_lots", y_test, mu_pois_feat, preds["project_id"].values, preds["total_lots"].values, dist_kind="poisson"
        )
    )
    if collect_weights:
        register_model_weight_columns(feat_feature_names, model_weight_feature_order, model_weight_feature_seen)
        model_weight_rows.append(
            collect_model_weights_wide(
                pois_feat,
                "Poisson_feature_remaining_lots",
                train_class,
                feat_feature_names,
            )
        )

    bb_res, bb_kappa, bb_fit_name = fit_betabinomial_glm_safe(
        y_train, n_train_bb, x_train_off, label="BetaBinomial"
    )
    print(f"[INFO] BetaBinomial used {bb_fit_name}")
    mu_bb = np.asarray(np.clip(bb_res.predict_mu(x_test_off, n_test_bb), 1e-9, None), dtype=float)
    preds["pred_betabinomial"] = mu_bb
    metrics.append(
        collect_metrics(
            "BetaBinomial",
            y_test,
            mu_bb,
            preds["project_id"].values,
            preds["total_lots"].values,
            dist_kind="betabinomial",
            n_trials=n_test_bb,
            kappa=bb_kappa,
        )
    )
    if collect_weights:
        register_model_weight_columns(off_feature_names, model_weight_feature_order, model_weight_feature_seen)
        model_weight_rows.append(
            collect_model_weights_wide(
                bb_res,
                "BetaBinomial",
                train_class,
                off_feature_names,
                kappa=bb_kappa,
            )
        )

    metrics_df = pd.DataFrame(metrics)
    if not metrics_df.empty:
        metrics_df["_abs_me_per_lot"] = metrics_df["ME_per_lot"].abs()
        metrics_df = metrics_df.sort_values(["_abs_me_per_lot", "chi2_distribution_per_bin"], na_position="last").drop(columns="_abs_me_per_lot").reset_index(drop=True)

    class_rows = []
    pred_map = {
        "NB_offset": ("pred_nb_offset", "nb", alpha_nb_off, None, None),
        "Poisson_offset": ("pred_poisson_offset", "poisson", None, None, None),
        "NB_feature_remaining_lots": ("pred_nb_feature_remaining_lots", "nb", alpha_nb_feat, None, None),
        "Poisson_feature_remaining_lots": ("pred_poisson_feature_remaining_lots", "poisson", None, None, None),
        "BetaBinomial": ("pred_betabinomial", "betabinomial", None, n_test_bb, bb_kappa),
    }

    for cls, sub in preds.groupby("class_group"):
        y_cls = sub["y_true"].to_numpy(dtype=float)
        idx = sub.index.to_numpy()
        for name, (col, dist_kind, alpha, n_trials_full, kappa) in pred_map.items():
            row = collect_metrics(
                name,
                y_cls,
                sub[col].to_numpy(dtype=float),
                sub["project_id"].values,
                sub["total_lots"].values,
                dist_kind=dist_kind,
                alpha=alpha,
                n_trials=(n_trials_full[idx] if n_trials_full is not None else None),
                kappa=kappa,
            )
            row["class_group"] = cls
            row["rows"] = len(sub)
            class_rows.append(row)

    class_metrics_df = pd.DataFrame(class_rows)
    if not class_metrics_df.empty:
        class_metrics_df["_abs_me_per_lot"] = class_metrics_df["ME_per_lot"].abs()
        class_metrics_df = class_metrics_df[["class_group", "rows", "model", "ME_per_lot", "chi2_distribution_per_bin", "chi2_bins_used", "_abs_me_per_lot"]]
        class_metrics_df = class_metrics_df.sort_values(["class_group", "_abs_me_per_lot", "chi2_distribution_per_bin"]).drop(columns="_abs_me_per_lot").reset_index(drop=True)

    return metrics_df, class_metrics_df, preds


## 3) Блок A — одна общая модель для всех классов сразу

`class_group` используется как обычная категориальная фича.  
Разбиение — по времени, но граница 60/40 выбирается по накопленной доле строк.

Для устойчивости NB в дизайне оставлены только более интерпретируемые и менее шумные признаки:
- возраст проекта;
- кварталы до РВЭ;
- доля нераспроданного остатка;
- лог цены за кв.м.;
- лог масштаба проекта (`total_lots`);
- лог относительной цены к рынку;
- класс жилья;
- сезонность (`quarter_num`).

Разреженные и шумные признаки (`district`, `building_type`, `year`, ценовые gap-признаки и дублирующие долевые признаки) исключены.


In [8]:

train_df_all, test_df_all, train_q_all, test_q_all = chronological_split_60_40(model_df)

print("Общий блок:")
print("  train rows:", len(train_df_all), "test rows:", len(test_df_all))
print("  train quarters:", ", ".join(map(str, train_q_all[:5])), "..." if len(train_q_all) > 5 else "")
print("  test quarters:", ", ".join(map(str, test_q_all)))

model_weight_rows = []
model_weight_feature_order = []
model_weight_feature_seen = set()

metrics_all_df, metrics_all_by_class_df, preds_all_block = run_four_models(
    train_df_all,
    test_df_all,
    include_class_feature=True,
    train_class="all_classes",
    model_weight_rows=model_weight_rows,
    model_weight_feature_order=model_weight_feature_order,
    model_weight_feature_seen=model_weight_feature_seen,
)

show_table(
    metrics_all_df,
    "Блок A — метрики на test (общая модель по всем классам сразу)",
    {"ME_per_lot": 6, "chi2_distribution_per_bin": 3},
)

show_table(
    metrics_all_by_class_df,
    "Блок A — метрики по классам на test",
    {"ME_per_lot": 6, "chi2_distribution_per_bin": 3},
)


Общий блок:
  train rows: 1783 test rows: 1046
  train quarters: 2009Q3, 2009Q4, 2010Q1, 2010Q2, 2010Q3 ...
  test quarters: 2022Q4, 2023Q1, 2023Q2, 2023Q3, 2023Q4, 2024Q1, 2024Q2, 2024Q3, 2024Q4, 2025Q1, 2025Q2, 2025Q3, 2025Q4
[INFO] NB_offset used ridge_alphaNB=2.5_pen=1e-05
[INFO] NB_feature_remaining_lots used irls_alpha=2.5
[INFO] BetaBinomial used lbfgs_kappa=6.696_pen=1e-06

Блок A — метрики на test (общая модель по всем классам сразу)


,model,ME_per_lot,chi2_distribution_per_bin,chi2_bins_used
0,NB_feature_remaining_lots,-0.000412,1.959,155
1,BetaBinomial,0.009482,1.640,186
2,Poisson_offset,-0.010713,6.993,210
3,Poisson_feature_remaining_lots,-0.011450,15.897,211
4,NB_offset,0.030393,2.354,145



Блок A — метрики по классам на test


,class_group,rows,model,ME_per_lot,chi2_distribution_per_bin,chi2_bins_used
0,Бизнес,100,NB_feature_remaining_lots,-0.006706,1.231,19
1,Бизнес,100,BetaBinomial,0.008331,1.517,14
2,Бизнес,100,NB_offset,0.012525,1.229,19
3,Бизнес,100,Poisson_offset,-0.012791,2.397,41
4,Бизнес,100,Poisson_feature_remaining_lots,-0.015442,3.728,37
5,Комфорт,808,BetaBinomial,0.004954,1.929,153
6,Комфорт,808,NB_feature_remaining_lots,-0.006795,2.266,131
7,Комфорт,808,Poisson_offset,-0.015590,2.855,203
8,Комфорт,808,Poisson_feature_remaining_lots,-0.015679,6.915,196
9,Комфорт,808,NB_offset,0.027326,2.663,119


## 4) Блок B — отдельная модель внутри каждого класса

Здесь для каждого класса жилья строится своя модель.  
Внутри класса признак `class_group` из дизайна убирается, потому что он константный.
На выходе оставляем только таблицу по классам.


In [9]:

per_class_tables = []

for cls, sub_df in model_df.groupby("class_group"):
    sub_df = sub_df.sort_values(["quarter", "project_id"]).copy()
    if sub_df["quarter"].nunique() < 2 or len(sub_df) < 10:
        continue

    train_cls, test_cls, train_q_cls, test_q_cls = chronological_split_60_40(sub_df)
    if len(train_cls) == 0 or len(test_cls) == 0:
        continue

    metrics_cls_df, _, _ = run_four_models(
        train_cls,
        test_cls,
        include_class_feature=False,
        train_class=str(cls),
        model_weight_rows=model_weight_rows,
        model_weight_feature_order=model_weight_feature_order,
        model_weight_feature_seen=model_weight_feature_seen,
    )

    metrics_cls_df["class_group"] = cls
    metrics_cls_df["rows_train"] = len(train_cls)
    metrics_cls_df["rows_test"] = len(test_cls)
    metrics_cls_df["train_quarters"] = ", ".join(map(str, train_q_cls))
    metrics_cls_df["test_quarters"] = ", ".join(map(str, test_q_cls))
    per_class_tables.append(metrics_cls_df)

metrics_separate_by_class_df = pd.concat(per_class_tables, ignore_index=True) if per_class_tables else pd.DataFrame()

if not metrics_separate_by_class_df.empty:
    metrics_separate_by_class_df["_abs_me_per_lot"] = metrics_separate_by_class_df["ME_per_lot"].abs()
    metrics_separate_by_class_df = metrics_separate_by_class_df[
        ["class_group", "model", "rows_train", "rows_test", "ME_per_lot", "chi2_distribution_per_bin", "chi2_bins_used", "train_quarters", "test_quarters", "_abs_me_per_lot"]
    ].sort_values(["class_group", "_abs_me_per_lot", "chi2_distribution_per_bin"]).drop(columns="_abs_me_per_lot").reset_index(drop=True)

show_table(
    metrics_separate_by_class_df,
    "Блок B — отдельные модели внутри каждого класса",
    {"ME_per_lot": 6, "chi2_distribution_per_bin": 3},
)

weights_df = build_weights_wide_df(model_weight_rows, model_weight_feature_order)
weights_df.to_csv("model_weights_summary.csv", index=False)
print("Saved weights to:", Path("model_weights_summary.csv").resolve())
show_table(weights_df.head(20), "Первые строки таблицы весов моделей")


[INFO] NB_offset used ridge_alphaNB=4.474_pen=1e-05
[INFO] NB_feature_remaining_lots used ridge_alphaNB=2.237_pen=1e-05
[INFO] BetaBinomial used lbfgs_kappa=15.71_pen=1e-06
[INFO] NB_offset used ridge_alphaNB=2.5_pen=1e-05
[INFO] NB_feature_remaining_lots used irls_alpha=2.5
[INFO] BetaBinomial used lbfgs_kappa=7.852_pen=1e-05
[INFO] NB_offset used irls_alpha=2.5
[INFO] NB_feature_remaining_lots used irls_alpha=2.5
[INFO] BetaBinomial used lbfgs_kappa=5.517_pen=0.0001

Блок B — отдельные модели внутри каждого класса


,class_group,model,rows_train,rows_test,ME_per_lot,chi2_distribution_per_bin,chi2_bins_used,train_quarters,test_quarters
0,Бизнес,Poisson_feature_remaining_lots,189,121,0.010415,1.054,32,"2013Q4, 2014Q1, 2014Q2, 2014Q3, 2014Q4, 2015Q1...","2022Q2, 2022Q3, 2022Q4, 2023Q1, 2023Q2, 2023Q3..."
1,Бизнес,Poisson_offset,189,121,0.019518,4.338,43,"2013Q4, 2014Q1, 2014Q2, 2014Q3, 2014Q4, 2015Q1...","2022Q2, 2022Q3, 2022Q4, 2023Q1, 2023Q2, 2023Q3..."
2,Бизнес,BetaBinomial,189,121,0.025122,1.787,18,"2013Q4, 2014Q1, 2014Q2, 2014Q3, 2014Q4, 2015Q1...","2022Q2, 2022Q3, 2022Q4, 2023Q1, 2023Q2, 2023Q3..."
3,Бизнес,NB_feature_remaining_lots,189,121,0.607396,1.956,16,"2013Q4, 2014Q1, 2014Q2, 2014Q3, 2014Q4, 2015Q1...","2022Q2, 2022Q3, 2022Q4, 2023Q1, 2023Q2, 2023Q3..."
4,Бизнес,NB_offset,189,121,3.551000,4.086,11,"2013Q4, 2014Q1, 2014Q2, 2014Q3, 2014Q4, 2015Q1...","2022Q2, 2022Q3, 2022Q4, 2023Q1, 2023Q2, 2023Q3..."
5,Комфорт,BetaBinomial,1129,660,0.003825,1.620,137,"2013Q2, 2013Q3, 2013Q4, 2014Q1, 2014Q2, 2014Q3...","2023Q2, 2023Q3, 2023Q4, 2024Q1, 2024Q2, 2024Q3..."
6,Комфорт,NB_feature_remaining_lots,1129,660,0.005058,1.975,114,"2013Q2, 2013Q3, 2013Q4, 2014Q1, 2014Q2, 2014Q3...","2023Q2, 2023Q3, 2023Q4, 2024Q1, 2024Q2, 2024Q3..."
7,Комфорт,Poisson_feature_remaining_lots,1129,660,-0.008312,2.478,198,"2013Q2, 2013Q3, 2013Q4, 2014Q1, 2014Q2, 2014Q3...","2023Q2, 2023Q3, 2023Q4, 2024Q1, 2024Q2, 2024Q3..."
8,Комфорт,Poisson_offset,1129,660,-0.009837,1.649,212,"2013Q2, 2013Q3, 2013Q4, 2014Q1, 2014Q2, 2014Q3...","2023Q2, 2023Q3, 2023Q4, 2024Q1, 2024Q2, 2024Q3..."
9,Комфорт,NB_offset,1129,660,0.072040,2.555,100,"2013Q2, 2013Q3, 2013Q4, 2014Q1, 2014Q2, 2014Q3...","2023Q2, 2023Q3, 2023Q4, 2024Q1, 2024Q2, 2024Q3..."


Saved weights to: C:\proga\gazprom_tex\cashflow\model_weights_summary.csv

Первые строки таблицы весов моделей


,model_name,train_class,const,project_age_q,quarters_to_delivery,remaining_inventory_share,log_avg_price_sqm_q,log_total_lots,log_price_to_market_year_ratio,class_group_Комфорт,class_group_Эконом,class_group_Элитный,quarter_num_2,quarter_num_3,quarter_num_4,log_remaining_lots_feat,__alpha__,__kappa__
0,NB_offset,all_classes,-3.794249,-0.380008,-0.784255,-1.311769,-0.349967,-0.458251,0.173840,0.171975,-0.133441,1.173314,0.303672,0.305622,-0.044457,NaN,2.500000,NaN
1,Poisson_offset,all_classes,-4.243110,-0.181812,-0.790346,-0.556105,-0.282354,-0.378248,0.238347,0.382607,0.095274,1.412594,0.122455,0.268079,0.029862,NaN,NaN,NaN
2,NB_feature_remaining_lots,all_classes,2.770488,-0.144013,-0.902037,0.052287,0.002316,0.696324,-0.037497,0.437899,0.317682,0.805645,0.171882,0.236067,0.027378,-0.148425,2.500000,NaN
3,Poisson_feature_remaining_lots,all_classes,2.673362,-0.200356,-0.850932,-0.093387,-0.214967,0.805000,0.235350,0.494061,0.239981,1.435422,0.111239,0.256581,0.022907,0.057255,NaN,NaN
4,BetaBinomial,all_classes,-3.296625,-0.164876,-0.517113,-0.584015,-0.074806,-0.080416,-0.045745,0.017606,-0.259802,1.203589,0.209896,0.194103,0.122405,NaN,NaN,6.695737
5,NB_offset,Бизнес,-5.644862,1.417569,-0.736538,-1.231044,0.549591,-1.397710,-0.046293,NaN,NaN,NaN,1.578230,1.370328,0.558711,NaN,4.474356,NaN
6,Poisson_offset,Бизнес,-5.050095,0.475834,-0.526562,-0.728786,-0.200090,-0.526588,0.195565,NaN,NaN,NaN,0.963309,0.440428,0.135082,NaN,NaN,NaN
7,NB_feature_remaining_lots,Бизнес,1.069863,1.405066,-0.842578,0.213327,0.626448,-0.230249,-0.131515,NaN,NaN,NaN,1.430597,1.125551,0.523122,-0.324953,2.237178,NaN
8,Poisson_feature_remaining_lots,Бизнес,1.760431,0.631240,-0.520559,0.081863,-0.057403,0.572974,0.109535,NaN,NaN,NaN,0.946817,0.441674,0.173491,-0.247823,NaN,NaN
9,BetaBinomial,Бизнес,-4.508842,0.372328,-0.347902,-1.099765,-0.126285,-0.434752,0.207692,NaN,NaN,NaN,0.426151,0.437453,0.143879,NaN,NaN,15.714544
